In [8]:
import pandas as pd
from scipy.stats import spearmanr, kendalltau

def pure_python_rbo(list1, list2, p=0.9):
    """
    Calculate extrapolated Rank-Biased Overlap (RBO) for two finite rankings.
    """
    s = set()
    t = set()
    rbo_sum = 0.0

    for i in range(len(list1)):
        s.add(list1[i])
        t.add(list2[i])

        overlap = len(s.intersection(t))
        agreement = overlap / (i + 1)

        rbo_sum += (p ** i) * agreement

    # Extrapolation term for finite rankings
    return (1 - p) * rbo_sum + (p ** len(list1)) * agreement

def bereken_macro_correlaties(orig_ranking, synth_ranking, p_value=0.9):
    """
    Berekent Spearman, Kendall en RBO tussen twee model-rankings.
    """
    model_to_pos = {model: i for i, model in enumerate(orig_ranking)}

    orig_pos = [model_to_pos[m] for m in orig_ranking]
    synth_pos = [model_to_pos[m] for m in synth_ranking]

    rho, _ = spearmanr(orig_pos, synth_pos)
    tau, _ = kendalltau(orig_pos, synth_pos)

    # Gebruik de pure python functie
    rbo_val = pure_python_rbo(orig_ranking, synth_ranking, p=p_value)

    return rho, tau, rbo_val

# --- HIERONDER BLIJFT DE REST VAN JE SCRIPT EXACT HETZELFDE ---
# 1. DutchNewsArticlesRetrieval
news_orig  = ['e5-large-trm', 'bge-m3', 'e5-small-trm', 'mBERT-cased-base', 'RobBERT-2023-base']
news_synth = ['e5-large-trm', 'bge-m3', 'e5-small-trm', 'mBERT-cased-base', 'RobBERT-2023-base']

# 2. OpenTenderRetrieval
tender_orig  = ['bge-m3', 'e5-large-trm', 'e5-small-trm', 'mBERT-cased-base', 'RobBERT-2023-base']
tender_synth = ['e5-large-trm', 'bge-m3', 'e5-small-trm', 'RobBERT-2023-base', 'mBERT-cased-base']

# 3. VABBRetrieval (Abstracts)
vabb_orig  = ['bge-m3', 'e5-large-trm', 'e5-small-trm', 'mBERT-cased-base', 'RobBERT-2023-base']
vabb_synth = ['e5-large-trm', 'bge-m3', 'e5-small-trm', 'RobBERT-2023-base', 'mBERT-cased-base']

datasets = {
    'DutchNewsArticlesRetrieval': (news_orig, news_synth),
    'OpenTenderRetrieval': (tender_orig, tender_synth),
    'VABBRetrieval': (vabb_orig, vabb_synth)
}

resultaten = []
for naam, (orig, synth) in datasets.items():
    rho, tau, rbo_score = bereken_macro_correlaties(orig, synth, p_value=0.9)
    resultaten.append({
        'Dataset': naam,
        'Spearman Rho': round(rho, 4),
        'Kendall Tau': round(tau, 4),
        'RBO (p=0.9)': round(rbo_score, 4)
    })

df = pd.DataFrame(resultaten)
print("\n=== MACRO RANKING CORRELATIE RESULTATEN ===")
print(df.to_string(index=False))


=== MACRO RANKING CORRELATIE RESULTATEN ===
                   Dataset  Spearman Rho  Kendall Tau  RBO (p=0.9)
DutchNewsArticlesRetrieval           1.0          1.0       1.0000
       OpenTenderRetrieval           0.8          0.6       0.8818
             VABBRetrieval           0.8          0.6       0.8818
